In [11]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')



normal_cond = (
    # 키(cm) : 150 ~ 190
    train["키(cm)"].between(150, 190)
    &
    # 몸무게(kg) : 45 ~ 90
    train["몸무게(kg)"].between(45, 90)
    &
    # BMI : 18.5 ~ 22.9
    train["BMI"].between(18.5, 22.9)
    &
    # 시력 : 1.0 이상  
    (train["시력"] >= 1.0)
    &
    # 공복 혈당 : 70 ~ 99 (mg/dL)
    train["공복 혈당"].between(70, 99)
    &
    # 이완기혈압 : 60 ~ 79 (mmHg)
    train["혈압"].between(60, 79)
    &
    # 중성 지방 : 30 ~ 149 (mg/dL)
    train["중성 지방"].between(30, 149)
    &
    # 혈청 크레아티닌 : 0.6 ~ 1.3 (남/녀 통합 범위로 사용)
    train["혈청 크레아티닌"].between(0.6, 1.3)
    &
    # 콜레스테롤(총) : 125 ~ 199 (mg/dL)
    train["콜레스테롤"].between(125, 199)
    &
    # 고밀도지단백(HDL) : 40 이상 (남자≥40, 여자≥50 → 완화해서 40 이상으로 잡음)
    (train["고밀도지단백"] >= 40)
    &
    # 저밀도지단백(LDL) : < 100  (0~100 구간으로 사용)
    train["저밀도지단백"].between(0, 100)
    &
    # 헤모글로빈 : 12 ~ 17 (여 12~16, 남 13~17 → 통합 범위)
    train["헤모글로빈"].between(12, 17)
    &
    # 요 단백 : 0 (음성)
    (train["요 단백"] == 0)
    &
    # 간 효소율(AST/ALT) : 10 ~ 40 (U/L 정도로 가정)
    train["간 효소율"].between(10, 40)
)

# 3. label==1 이면서 위 "정상 수치 조건"에 모두 해당하는 행 찾기
mask_drop = (train["label"] == 1) & normal_cond
none_mask_drop = (train["label"] == 0) & normal_cond

 
num_healthy_none = none_mask_drop.sum()
num_healthy_smokers = mask_drop.sum()

print(f"정상수치이면서 흡연자인 사람 수: {num_healthy_smokers}명")
print(f"정상수치이면서 비흡연자인 사람 수: {num_healthy_none}명")

정상수치이면서 흡연자인 사람 수: 0명
정상수치이면서 비흡연자인 사람 수: 0명


In [7]:
normal_cond = (
    # 키(cm) : 150 ~ 190
    train["키(cm)"].between(150, 190)
    &
    # 몸무게(kg) : 45 ~ 90
    train["몸무게(kg)"].between(45, 90)
    &
    # BMI : 18.5 ~ 22.9
    train["BMI"].between(18.5, 22.9)
    &
    # 시력 : 1.0 이상  
    (train["시력"] >= 1.0)
    &
    # 공복 혈당 : 70 ~ 99 (mg/dL)
    train["공복 혈당"].between(70, 99)
    &
    # 이완기혈압 : 60 ~ 79 (mmHg)
    train["혈압"].between(60, 79)
    &
    # 중성 지방 : 30 ~ 149 (mg/dL)
    train["중성 지방"].between(30, 149)
    &
    # 혈청 크레아티닌 : 0.6 ~ 1.3 (남/녀 통합 범위로 사용)
    train["혈청 크레아티닌"].between(0.6, 1.3)
    &
    # 콜레스테롤(총) : 125 ~ 199 (mg/dL)
    train["콜레스테롤"].between(125, 199)
    &
    # 고밀도지단백(HDL) : 40 이상 (남자≥40, 여자≥50 → 완화해서 40 이상으로 잡음)
    (train["고밀도지단백"] >= 40)
    &
    # 저밀도지단백(LDL) : < 100  (0~100 구간으로 사용)
    train["저밀도지단백"].between(0, 100)
    &
    # 헤모글로빈 : 12 ~ 17 (여 12~16, 남 13~17 → 통합 범위)
    train["헤모글로빈"].between(12, 17)
    &
    # 요 단백 : 0 (음성)
    (train["요 단백"] == 0)
    &
    # 간 효소율(AST/ALT) : 10 ~ 40 (U/L 정도로 가정)
    train["간 효소율"].between(10, 40)
)

# 3. label==1 이면서 위 "정상 수치 조건"에 모두 해당하는 행 찾기
mask_drop = (train["label"] == 0) & normal_cond


# 4. 해당 행들을 제거한 새로운 train 데이터셋 만들기
train_clean = train.loc[~mask_drop].reset_index(drop=True)


print("원래 train shape :", train.shape)
print("정상수치이면서 비흡연자인 train shape:", train_clean.shape)

원래 train shape : (7000, 18)
정상수치이면서 비흡연자인 train shape: (7000, 18)
